In [1]:
import numpy as np

In [2]:
import pandas as pd

In [3]:
d=pd.read_csv('/kaggle/input/competitions/spaceship-titanic/sample_submission.csv')

In [4]:
d.head()

,PassengerId,Transported
0,0013_01,False
1,0018_01,False
2,0019_01,False
3,0021_01,False
4,0023_01,False


In [5]:
from sklearn.preprocessing import LabelEncoder

In [6]:
le=LabelEncoder()

In [7]:
d['Transported']=le.fit_transform(d['Transported'])

In [8]:
d.head()

,PassengerId,Transported
0,0013_01,0
1,0018_01,0
2,0019_01,0
3,0021_01,0
4,0023_01,0


In [9]:
d.shape

(4277, 2)

In [10]:
d['Transported'].value_counts()

Transported
0    4277
Name: count, dtype: int64

In [11]:
train=pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')

In [12]:
test=pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')

In [13]:
train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [14]:
train['Transported']=le.fit_transform(train['Transported'])

In [15]:
train['Transported'].value_counts()

Transported
1    4378
0    4315
Name: count, dtype: int64

In [16]:
train.shape

(8693, 14)

In [17]:
train.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [18]:
train=train.drop(columns=['PassengerId','HomePlanet','Destination','Cabin','Name'])

In [19]:
Y=train['Transported']

In [20]:
from sklearn.impute import SimpleImputer

In [21]:
si=SimpleImputer(strategy='median')

In [22]:
pd.set_option('display.max_rows',None)

In [23]:
train.isnull().sum()

CryoSleep       217
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Transported       0
dtype: int64

In [24]:
train['VIP']=le.fit_transform(train['VIP'])

In [25]:
train['CryoSleep']=le.fit_transform(train['CryoSleep'])

In [26]:
from sklearn.model_selection import train_test_split

In [27]:
train.isnull().sum()

CryoSleep         0
Age             179
VIP               0
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Transported       0
dtype: int64

In [28]:
X=train.drop(columns='Transported')

In [29]:
Y=train['Transported']

In [30]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,stratify=Y,random_state=42,test_size=0.2)

In [31]:
X_train=si.fit_transform(X_train)

In [32]:
X_test=si.fit_transform(X_test)

In [33]:
Y_train.shape

(6954,)

In [34]:
X_train.shape

(6954, 8)

In [35]:
from xgboost import XGBClassifier
     

xgb=XGBClassifier()
     

estimators=[('xgb',xgb)]
     

from sklearn.pipeline import Pipeline
     

pipe=Pipeline(steps=estimators)
     

pipe

Pipeline(steps=[('xgb',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric=None,
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [36]:
from skopt import BayesSearchCV
     

from skopt.space import Real,Categorical,Integer
     

search_space = {
    'max_depth': Integer(3, 8),
    'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    'n_estimators': Integer(100, 500),
    'subsample': Real(0.6, 1.0),
    'colsample_bytree': Real(0.6, 1.0),
    'reg_alpha': Real(0.0, 5.0),
    'reg_lambda': Real(0.1, 5.0),
    'gamma': Real(0.0, 5.0)
}

In [37]:
opt=BayesSearchCV(pipe,search_space,n_iter=10,cv=3,scoring='roc_auc',random_state=8)
     

opt = BayesSearchCV(
    xgb,
    search_spaces=search_space,
    n_iter=10,
    cv=3
)

In [38]:
opt.fit(X_train,Y_train)

BayesSearchCV(cv=3,
              estimator=XGBClassifier(base_score=None, booster=None,
                                      callbacks=None, colsample_bylevel=None,
                                      colsample_bynode=None,
                                      colsample_bytree=None, device=None,
                                      early_stopping_rounds=None,
                                      enable_categorical=False,
                                      eval_metric=None, feature_types=None,
                                      feature_weights=None, gamma=None,
                                      grow_policy=None, importance_type=None,
                                      interaction_constraints=No...
                             'max_depth': Integer(low=3, high=8, prior='uniform', transform='normalize'),
                             'n_estimators': Integer(low=100, high=500, prior='uniform', transform='normalize'),
                             'reg_alpha': Real(low=0.0, high=5.0, prior='uniform', transform='normalize'),
                             'reg_lambda': Real(low=0.1, high=5.0, prior='uniform', transform='normalize'),
                             'subsample': Real(low=0.6, high=1.0, prior='uniform', transform='normalize')})

In [39]:
Y1pred=opt.predict(X_test)

In [40]:
from sklearn.metrics import accuracy_score,classification_report

In [41]:
accuracy_score(Y_test,Y1pred)

0.8016101207590569

In [42]:
classification_report(Y_test,Y1pred)

'              precision    recall  f1-score   support\n\n           0       0.82      0.76      0.79       863\n           1       0.78      0.84      0.81       876\n\n    accuracy                           0.80      1739\n   macro avg       0.80      0.80      0.80      1739\nweighted avg       0.80      0.80      0.80      1739\n'